# LeetCode 967 — Numbers With Same Consecutive Differences bằng Python

Link đề: https://leetcode.com/problems/numbers-with-same-consecutive-differences/ · Độ khó: **Medium** · Chủ đề: Backtracking (quay lui), BFS (duyệt theo tầng), DFS (duyệt theo chiều sâu)

Skeleton LeetCode:

```python
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        ...
```

Ghi chú: liệt kê **mọi số nguyên có đúng `n` chữ số** (không có số 0 đứng đầu) sao cho **hiệu tuyệt đối giữa hai chữ số liền kề luôn bằng `k`**. Thứ tự kết quả **không quan trọng** (đề cho trả về theo thứ tự bất kỳ), nên khi kiểm thử ta **sort** hai danh sách trước khi so sánh. Cạm bẫy chính là `k = 0` (hai nhánh `+k` và `-k` trùng nhau nên sinh **số lặp**) và số 0 đứng đầu.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sinh mọi dãy `n` chữ số mà hai chữ số kề nhau chênh đúng `k`, chữ số đầu khác `0`.
2. Nắm các công cụ Python cần thiết: đệ quy (DFS), hàng đợi theo tầng (BFS bằng list), `set` để khử trùng nhánh, `divmod`/`% 10` để lấy chữ số cuối, và `sorted(...)` để so sánh danh sách không có thứ tự.
3. Hiểu nguyên lý: chữ số tiếp theo chỉ có **tối đa hai lựa chọn** (`last - k` và `last + k`), nên số kết quả tối đa là `9 * 2^(n-1)`: nhỏ, và quay lui hiệu quả.
4. Đi qua 4 cách: duyệt mọi số `n` chữ số rồi kiểm tra (brute force, chỉ dùng cho `n` nhỏ), DFS/quay lui (cách nộp chính), BFS theo tầng, và quy hoạch động theo chữ số đầu (dùng lại kết quả con).
5. Tránh các lỗi kinh điển: `k = 0` sinh số lặp, cho phép số 0 đứng đầu, quên kiểm tra biên `0..9`, chỉ thử một hướng `+k`, và bỏ sót chữ số đầu `9` (vòng lặp `range(1, 9)`).

## 2. Đề bài & ví dụ

Cho hai số nguyên `n` và `k`. Trả về một mảng gồm **tất cả các số nguyên có độ dài `n`** mà **hiệu giữa hai chữ số liền kề bằng `k`**. Bạn có thể trả về đáp án theo **thứ tự bất kỳ**.

Lưu ý: các số **không được có số 0 đứng đầu**. Chẳng hạn `02` và `043` không hợp lệ.

**Ví dụ 1:** `n = 3, k = 7` -> `[181,292,707,818,929]`. Chú ý `070` không hợp lệ vì có số 0 đứng đầu.

**Ví dụ 2:** `n = 2, k = 1` -> `[10,12,21,23,32,34,43,45,54,56,65,67,76,78,87,89,98]`.

**Ràng buộc:**

- `2 <= n <= 9`
- `0 <= k <= 9`

In [1]:
from typing import List, Set, Tuple
from itertools import product
from collections import deque
import random, time

examples = [
    (3, 7, [181, 292, 707, 818, 929]),
    (2, 1, [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 76, 78, 87, 89, 98]),
]

def same_set(a: List[int], b: List[int]) -> bool:
    """Thứ tự không quan trọng: so sánh sau khi sort (và danh sách phải đúng số phần tử, không trùng)."""
    return sorted(a) == sorted(b)

for n, k, expected in examples:
    print(f"n={n} k={k} -> {len(expected)} số, kỳ vọng {expected}")

n=3 k=7 -> 5 số, kỳ vọng [181, 292, 707, 818, 929]
n=2 k=1 -> 17 số, kỳ vọng [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 76, 78, 87, 89, 98]


## 3. Kiến thức nền

### 3.1. Chữ số cuối và ghép thêm một chữ số

Với số nguyên dương `num`:

- chữ số cuối: `num % 10`;
- thêm chữ số `d` vào cuối: `num * 10 + d`.

Nhờ vậy ta xây số dần dần mà không cần chuyển qua chuỗi.

In [2]:
num = 18
print("chữ số cuối của 18:", num % 10, "| thêm chữ số 1:", num * 10 + 1)
print("tách bằng divmod:", divmod(181, 10))

chữ số cuối của 18: 8 | thêm chữ số 1: 181
tách bằng divmod: (18, 1)


### 3.2. Hai lựa chọn: `last - k` và `last + k`

Nếu chữ số cuối là `last`, chữ số kế tiếp `d` phải thỏa `|d - last| = k`, tức `d = last + k` hoặc `d = last - k`. Cả hai phải nằm trong `0..9`. Khi `k = 0` hai giá trị trùng nhau (`last`): **chỉ được tính một lần**. Dùng `set({last - k, last + k})` khử trùng tự động.

In [3]:
for last, k in [(1, 7), (8, 7), (5, 0), (4, 9), (0, 3)]:
    cand = {last - k, last + k}
    print(f"last={last} k={k}: ứng viên {sorted(cand)} -> hợp lệ {sorted(d for d in cand if 0 <= d <= 9)}")

last=1 k=7: ứng viên [-6, 8] -> hợp lệ [8]
last=8 k=7: ứng viên [1, 15] -> hợp lệ [1]
last=5 k=0: ứng viên [5] -> hợp lệ [5]
last=4 k=9: ứng viên [-5, 13] -> hợp lệ []
last=0 k=3: ứng viên [-3, 3] -> hợp lệ [3]


### 3.3. Thứ tự kết quả không quan trọng

Đề cho phép trả về theo **thứ tự bất kỳ**. Các cách khác nhau cho ra thứ tự khác nhau (DFS, BFS...), nên khi kiểm thử ta so sánh `sorted(a) == sorted(b)`. Việc sort **giữ nguyên số lần xuất hiện**, nên nếu có số bị lặp (lỗi `k = 0`) thì phép so sánh sẽ phát hiện ra.

In [4]:
print(sorted([3, 1, 2]) == sorted([2, 3, 1]), "| có lặp:", sorted([1, 1, 2]) == sorted([1, 2]))

True | có lặp: False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `n = 3`, `k = 7`. Chữ số đầu từ `1..9`; mỗi bước chọn `last ± 7` trong `0..9`.

- Đầu `1`: kế `1 + 7 = 8` (còn `1 - 7 = -6` loại); sau `8`: `8 + 7 = 15` loại, `8 - 7 = 1` -> `181`.
- Đầu `2`: kế `9`; sau `9`: `9 - 7 = 2` -> `292`.
- Đầu `3`: `3 + 7 = 10` loại, `3 - 7 < 0` loại -> không có số nào.
- Đầu `7`: kế `0` (`7 - 7`) hoặc `14` loại; sau `0`: `0 + 7 = 7` -> `707`.
- Đầu `8`: kế `1`; sau `1`: `8` -> `818`. Đầu `9`: kế `2`; sau `2`: `9` -> `929`.

Kết quả: `181, 292, 707, 818, 929` (5 số). Số `070` không được tính vì ta **không bắt đầu bằng `0`**.

## 5. Cách 1 — Brute Force: duyệt mọi số `n` chữ số rồi kiểm tra

**Ý tưởng.** Mọi số `n` chữ số nằm trong `[10^(n-1), 10^n - 1]`. Với mỗi số, chuyển sang chuỗi và kiểm tra `|d[i] - d[i+1]| == k` với mọi `i`. Đây là **định nghĩa trực tiếp** nên cũng là **oracle độc lập** (không dùng quay lui, không dùng `last ± k`).

In [5]:
def nums_brute(n: int, k: int) -> List[int]:
    res = []
    for x in range(10 ** (n - 1), 10 ** n):
        s = str(x)
        if all(abs(int(s[i]) - int(s[i + 1])) == k for i in range(n - 1)):
            res.append(x)
    return res

for n, k, expected in examples:
    got = nums_brute(n, k)
    print(f"n={n} k={k}:", "OK" if same_set(got, expected) else "SAI", got)
    assert same_set(got, expected)

n=3 k=7: OK [181, 292, 707, 818, 929]
n=2 k=1: OK [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 76, 78, 87, 89, 98]


**Độ phức tạp và nút thắt.** `O(9 * 10^(n-1) * n)`: với `n = 9` có `9 * 10^8` số: không thể. Đo thử rồi ngoại suy:

In [6]:
for n in (3, 4, 5, 6):
    t0 = time.perf_counter()
    nums_brute(n, 1)
    dt = time.perf_counter() - t0
    print(f"n={n}: {dt * 1000:9.1f} ms  -> ngoại suy n=9: ~{dt * 10 ** (9 - n):10.0f} s")

n=3:       0.4 ms  -> ngoại suy n=9: ~       384 s
n=4:       3.9 ms  -> ngoại suy n=9: ~       388 s
n=5:      40.1 ms  -> ngoại suy n=9: ~       401 s


n=6:     467.9 ms  -> ngoại suy n=9: ~       468 s


Nút thắt: ta duyệt cả những số **chắc chắn sai** (ví dụ chữ số thứ hai đã lệch). Nên **cắt tỉa**: xây số từng chữ số và chỉ mở rộng bằng `last ± k` hợp lệ.

## 6. Ý tưởng thuật toán / bước đột phá

**Xây từng chữ số, mỗi bước chỉ có tối đa 2 lựa chọn.** Từ chữ số cuối `last`, chữ số kế phải là `last + k` hoặc `last - k` (thuộc `0..9`). Do đó số kết quả `<= 9 * 2^(n-1)` (với `n = 9`: tối đa `2304`): nhỏ, ta chỉ việc **sinh trực tiếp** các số hợp lệ, không bao giờ chạm tới số sai.

Ba cách cài đặt sinh đó:

| Cách | Ý tưởng |
|---|---|
| 2 | DFS (quay lui): đi sâu đến đủ `n` chữ số rồi ghi nhận (nộp) |
| 3 | BFS: mở rộng đồng loạt từng tầng (mỗi tầng thêm một chữ số) |
| 4 | DP theo chiều dài: `S[m][d]` = các số `m` chữ số bắt đầu bằng `d`, dùng lại `S[m-1]` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS / quay lui (cách nộp chính)

**Giải thích từng đoạn.**

- `dfs(num, last, left)`: đang có số `num` với chữ số cuối `last`, còn `left` chữ số phải thêm.
- `left == 0`: đã đủ `n` chữ số, ghi nhận `num`.
- Duyệt `d` trong `{last - k, last + k}` (tập hợp để khi `k = 0` chỉ còn một phần tử), nếu `0 <= d <= 9` thì `dfs(num * 10 + d, d, left - 1)`.
- Vòng ngoài thử chữ số đầu `1..9` (không cho `0`).

In [7]:
def nums_dfs(n: int, k: int) -> List[int]:
    res: List[int] = []

    def dfs(num: int, last: int, left: int) -> None:
        if left == 0:
            res.append(num)
            return
        for d in {last - k, last + k}:
            if 0 <= d <= 9:
                dfs(num * 10 + d, d, left - 1)

    for first in range(1, 10):
        dfs(first, first, n - 1)
    return res

for n, k, expected in examples:
    got = nums_dfs(n, k)
    print(f"n={n} k={k}:", "OK" if same_set(got, expected) else "SAI", got)
    assert same_set(got, expected)

n=3 k=7: OK [181, 292, 707, 818, 929]
n=2 k=1: OK [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 78, 76, 89, 87, 98]


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        res = []

        def dfs(num: int, last: int, left: int) -> None:
            if left == 0:
                res.append(num)
                return
            for d in {last - k, last + k}:      # k = 0: tập hợp chỉ còn một phần tử, không bị lặp
                if 0 <= d <= 9:
                    dfs(num * 10 + d, d, left - 1)

        for first in range(1, 10):
            dfs(first, first, n - 1)
        return res

sol_dfs = Solution().numsSameConsecDiff
for n, k, expected in examples:
    assert same_set(sol_dfs(n, k), expected)
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(9 * 2^(n-1) * n)` (số kết quả nhân độ dài); bộ nhớ ngoài kết quả: `O(n)` cho ngăn xếp đệ quy.

---

### 7.2. Cách 3 — BFS theo tầng

**Ý tưởng khác.** Thay vì đi sâu, ta giữ **danh sách các số hiện có** (tầng hiện tại, mọi số cùng số chữ số). Mỗi vòng, mở rộng từng số bằng `last ± k` để được tầng kế tiếp. Sau `n - 1` vòng, danh sách chính là đáp án.

**Giải thích từng đoạn.**

- Tầng 1: `[1, 2, ..., 9]`.
- `last = num % 10`; chữ số mới `d` từ `{last - k, last + k}` hợp lệ; số mới `num * 10 + d`.
- Không cần đệ quy (tránh giới hạn độ sâu, tuy ở đây `n <= 9` rất nhỏ).

In [9]:
def nums_bfs(n: int, k: int) -> List[int]:
    level = list(range(1, 10))
    for _ in range(n - 1):
        nxt = []
        for num in level:
            last = num % 10
            for d in {last - k, last + k}:
                if 0 <= d <= 9:
                    nxt.append(num * 10 + d)
        level = nxt
    return level

for n, k, expected in examples:
    got = nums_bfs(n, k)
    print(f"n={n} k={k}:", "OK" if same_set(got, expected) else "SAI", got)
    assert same_set(got, expected)

n=3 k=7: OK [181, 292, 707, 818, 929]
n=2 k=1: OK [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 78, 76, 89, 87, 98]


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        level = list(range(1, 10))
        for _ in range(n - 1):
            nxt = []
            for num in level:
                last = num % 10
                for d in {last - k, last + k}:
                    if 0 <= d <= 9:
                        nxt.append(num * 10 + d)
            level = nxt
        return level

sol_bfs = Solution().numsSameConsecDiff
for n, k, expected in examples:
    assert same_set(sol_bfs(n, k), expected)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(9 * 2^(n-1) * n)`; bộ nhớ `O(số kết quả)` cho tầng hiện tại (cùng bậc với đáp án).

---

### 7.3. Cách 4 — DP theo chiều dài (dùng lại kết quả con)

**Ý tưởng khác.** Gọi `S[m][d]` là danh sách các số `m` chữ số **bắt đầu bằng chữ số `d`** (cho phép `d = 0` ở các tầng con vì số con ở giữa có thể bắt đầu bằng 0). Công thức:

- `S[1][d] = [d]`;
- `S[m][d] = [d * 10^(m-1) + x for d' in {d - k, d + k} hợp lệ for x in S[m-1][d']]`.

Đáp án là ghép `S[n][d]` với `d = 1..9`. Mỗi `S[m][d]` được tính một lần rồi dùng lại (khác DFS vì DFS tính lại các nhánh con giống nhau).

**Giải thích từng đoạn.**

- `d * 10^(m-1)` đặt chữ số `d` ở vị trí cao nhất; phần còn lại `x` lấy từ `S[m-1][d']`.
- Ở các tầng con ta **cho phép chữ số đầu là `0`** (ví dụ phần đuôi `07` của số `707`): giá trị số học của `x` khi đó nhỏ hơn `10^(m-2)`, nhưng cộng `d * 10^(m-1)` vẫn cho đúng số `m` chữ số vì `d >= 1` ở tầng ngoài cùng.

In [11]:
def nums_dp(n: int, k: int) -> List[int]:
    S = {d: [d] for d in range(10)}                       # S[1][d] = [d]
    for m in range(2, n + 1):
        newS = {}
        for d in range(10):
            lst = []
            for nd in {d - k, d + k}:
                if 0 <= nd <= 9:
                    base = d * 10 ** (m - 1)
                    lst.extend(base + x for x in S[nd])
            newS[d] = lst
        S = newS
    return [x for d in range(1, 10) for x in S[d]]

for n, k, expected in examples:
    got = nums_dp(n, k)
    print(f"n={n} k={k}:", "OK" if same_set(got, expected) else "SAI", got)
    assert same_set(got, expected)

n=3 k=7: OK [181, 292, 707, 818, 929]
n=2 k=1: OK [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 78, 76, 89, 87, 98]


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        S = {d: [d] for d in range(10)}          # S[d]: các số m chữ số (theo giá trị) bắt đầu bằng d
        for m in range(2, n + 1):
            newS = {}
            for d in range(10):
                lst = []
                base = d * 10 ** (m - 1)
                for nd in {d - k, d + k}:
                    if 0 <= nd <= 9:
                        lst.extend(base + x for x in S[nd])
                newS[d] = lst
            S = newS
        return [x for d in range(1, 10) for x in S[d]]

sol_dp = Solution().numsSameConsecDiff
for n, k, expected in examples:
    assert same_set(sol_dp(n, k), expected)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian và bộ nhớ `O(10 * (số số ở các tầng con) )`, cùng bậc `O(2^n)` nhưng có hằng số lớn hơn vì lưu cả các số bắt đầu bằng `0` ở tầng con.

## 8. TRACE từng bước

### 8.1. DFS cho `n = 3, k = 7`

In từng lần gọi `dfs` (thụt lề theo độ sâu) để thấy các nhánh bị cắt.

In [13]:
def trace_dfs(n: int, k: int):
    res = []

    def dfs(num, last, left, depth):
        pad = "  " * depth
        if left == 0:
            print(f"{pad}-> đủ {n} chữ số: ghi nhận {num}")
            res.append(num)
            return
        opts = sorted({last - k, last + k})
        ok = [d for d in opts if 0 <= d <= 9]
        print(f"{pad}num={num} last={last}: ứng viên {opts} -> hợp lệ {ok}")
        for d in ok:
            dfs(num * 10 + d, d, left - 1, depth + 1)

    for first in range(1, 10):
        dfs(first, first, n - 1, 0)
    return res

got = trace_dfs(3, 7)
assert same_set(got, [181, 292, 707, 818, 929])

num=1 last=1: ứng viên [-6, 8] -> hợp lệ [8]
  num=18 last=8: ứng viên [1, 15] -> hợp lệ [1]
    -> đủ 3 chữ số: ghi nhận 181
num=2 last=2: ứng viên [-5, 9] -> hợp lệ [9]
  num=29 last=9: ứng viên [2, 16] -> hợp lệ [2]
    -> đủ 3 chữ số: ghi nhận 292
num=3 last=3: ứng viên [-4, 10] -> hợp lệ []
num=4 last=4: ứng viên [-3, 11] -> hợp lệ []
num=5 last=5: ứng viên [-2, 12] -> hợp lệ []
num=6 last=6: ứng viên [-1, 13] -> hợp lệ []
num=7 last=7: ứng viên [0, 14] -> hợp lệ [0]
  num=70 last=0: ứng viên [-7, 7] -> hợp lệ [7]
    -> đủ 3 chữ số: ghi nhận 707
num=8 last=8: ứng viên [1, 15] -> hợp lệ [1]
  num=81 last=1: ứng viên [-6, 8] -> hợp lệ [8]
    -> đủ 3 chữ số: ghi nhận 818
num=9 last=9: ứng viên [2, 16] -> hợp lệ [2]
  num=92 last=2: ứng viên [-5, 9] -> hợp lệ [9]
    -> đủ 3 chữ số: ghi nhận 929


### 8.2. BFS cho `n = 3, k = 7`: các tầng

Từng tầng là danh sách số có cùng số chữ số.

In [14]:
def trace_bfs(n: int, k: int):
    level = list(range(1, 10))
    print("tầng 1:", level)
    for step in range(2, n + 1):
        nxt = []
        for num in level:
            last = num % 10
            for d in sorted({last - k, last + k}):
                if 0 <= d <= 9:
                    nxt.append(num * 10 + d)
        level = nxt
        print(f"tầng {step}:", level)
    return level

got = trace_bfs(3, 7)
assert same_set(got, [181, 292, 707, 818, 929])

tầng 1: [1, 2, 3, 4, 5, 6, 7, 8, 9]
tầng 2: [18, 29, 70, 81, 92]
tầng 3: [181, 292, 707, 818, 929]


### 8.3. `k = 0`: vì sao cần khử trùng nhánh

Với `k = 0`, `last - k` và `last + k` đều bằng `last`. Nếu duyệt hai nhánh riêng (không dùng `set`) thì mỗi bước nhân đôi và sinh ra số lặp. Với `n = 3` đáp án đúng là `[111, 222, ..., 999]` (9 số), còn cách sai sinh `9 * 2^2 = 36` số.

In [15]:
def nums_dup(n, k):                       # cố ý sai: hai nhánh +k và -k riêng biệt
    res = []
    def dfs(num, last, left):
        if left == 0:
            res.append(num)
            return
        for d in (last - k, last + k):
            if 0 <= d <= 9:
                dfs(num * 10 + d, d, left - 1)
    for f in range(1, 10):
        dfs(f, f, n - 1)
    return res

print("đúng:", sorted(Solution().numsSameConsecDiff(3, 0)))
print("sai: ", len(nums_dup(3, 0)), "số, ví dụ lặp:", sorted(nums_dup(3, 0))[:4])
assert len(nums_dup(3, 0)) == 36 and len(Solution().numsSameConsecDiff(3, 0)) == 9

đúng: [111, 222, 333, 444, 555, 666, 777, 888, 999]
sai:  36 số, ví dụ lặp: [111, 111, 111, 111]


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (DFS / quay lui): ngắn gọn, dễ chứng minh đúng.

```python
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        res = []

        def dfs(num: int, last: int, left: int) -> None:
            if left == 0:
                res.append(num)
                return
            for d in {last - k, last + k}:      # k = 0: tập hợp chỉ còn một phần tử, không bị lặp
                if 0 <= d <= 9:
                    dfs(num * 10 + d, d, left - 1)

        for first in range(1, 10):
            dfs(first, first, n - 1)
        return res
```

In [16]:
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        res = []

        def dfs(num: int, last: int, left: int) -> None:
            if left == 0:
                res.append(num)
                return
            for d in {last - k, last + k}:      # k = 0: tập hợp chỉ còn một phần tử, không bị lặp
                if 0 <= d <= 9:
                    dfs(num * 10 + d, d, left - 1)

        for first in range(1, 10):
            dfs(first, first, n - 1)
        return res

sol = Solution()
for n, k, expected in examples:
    got = sol.numsSameConsecDiff(n, k)
    print(f"n={n} k={k}:", "OK" if same_set(got, expected) else "SAI", sorted(got))
    assert same_set(got, expected)

n=3 k=7: OK [181, 292, 707, 818, 929]
n=2 k=1: OK [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 76, 78, 87, 89, 98]


## 10. Vì sao đúng

**Bất biến.** Khi `dfs(num, last, left)` được gọi, `num` là một số có chữ số đầu khác `0`, các cặp chữ số liền kề của `num` đều chênh đúng `k`, và chữ số cuối là `last`; còn cần thêm `left` chữ số.

**Tính đầy đủ (không sót).** Mọi số hợp lệ `d1 d2 ... dn` có `d1 in 1..9` (vòng ngoài thử hết). Giả sử đã xây tiền tố `d1..di`; chữ số `d(i+1)` thỏa `|d(i+1) - di| = k` nên thuộc `{di - k, di + k}` và nằm trong `0..9`: nhánh này được duyệt. Bằng quy nạp, số hợp lệ được sinh ra.

**Tính đúng (không thừa, không lặp).** Mỗi số sinh ra thỏa điều kiện theo bất biến. Hai nhánh khác nhau của cùng một nút đi tới hai chữ số **khác nhau** (khi `k > 0`: `last - k != last + k`) hoặc bị gộp bằng `set` (khi `k = 0`). Hai tiền tố khác nhau không bao giờ sinh ra cùng một số, nên không có số lặp.

**Số 0 đứng đầu.** Chữ số đầu chỉ lấy từ `1..9`, nên không có số 0 đứng đầu. Số 0 vẫn có thể xuất hiện ở vị trí khác (`707`, `10`).

**Số kết quả.** Mỗi bước có `<= 2` lựa chọn, vậy có `<= 9 * 2^(n-1) = 2304` số khi `n = 9`: đúng với mọi `n <= 9`.

**Kiểm chứng bằng máy.** Oracle độc lập (Cách 1) duyệt **mọi** số `n` chữ số và kiểm tra theo định nghĩa. Ta so sánh sau khi sort cho **mọi** `k = 0..9` và `n = 2..5` (và `n = 6` cho vài `k`), rồi cho cả lưới `n = 2..9`, `k = 0..9` đối chiếu giữa các cách và với bộ đếm bằng quy hoạch động (độc lập với việc sinh số).

In [17]:
def count_dp(n: int, k: int) -> int:
    """Đếm (không sinh số): cnt[d] = số cách đi tiếp từ chữ số d; chữ số đầu 1..9."""
    cnt = [1] * 10
    for _ in range(n - 1):
        new = [0] * 10
        for d in range(10):
            for nd in {d - k, d + k}:
                if 0 <= nd <= 9:
                    new[d] += cnt[nd]
        cnt = new
    return sum(cnt[1:])

for n in range(2, 6):
    for k in range(10):
        ref = nums_brute(n, k)
        for name, fn in (("dfs", nums_dfs), ("bfs", nums_bfs), ("dp", nums_dp), ("nộp", sol.numsSameConsecDiff)):
            assert same_set(fn(n, k), ref), (name, n, k)
        assert count_dp(n, k) == len(ref), (n, k)
for k in (0, 1, 5, 9):
    ref = nums_brute(6, k)
    assert same_set(sol.numsSameConsecDiff(6, k), ref) and count_dp(6, k) == len(ref)
print("n = 2..5 mọi k, và n = 6 với k in (0, 1, 5, 9): bốn cách + bộ đếm khớp brute")

n = 2..5 mọi k, và n = 6 với k in (0, 1, 5, 9): bốn cách + bộ đếm khớp brute


## 11. Time / Space Complexity

| Cách | Time | Space (không tính đáp án) |
|---|---|---|
| 1. Duyệt mọi số | `O(9 * 10^(n-1) * n)` | `O(1)` |
| 2. DFS | `O(9 * 2^(n-1) * n)` | `O(n)` |
| 3. BFS | `O(9 * 2^(n-1) * n)` | `O(2^n)` (tầng hiện tại) |
| 4. DP theo chiều dài | `O(2^n * hệ số 10/9)` | `O(2^n)` |

Số kết quả lớn nhất (đo bằng máy, với mọi `n <= 9`, `k`) và thời gian của từng cách ở cỡ tối đa:

In [18]:
best = max(((count_dp(9, k), k) for k in range(10)))
print("số kết quả lớn nhất khi n = 9:", best, "| cận lý thuyết 9 * 2^8 =", 9 * 2 ** 8)
assert best[0] <= 9 * 2 ** 8

for k in (0, 1, 3, 5):
    for name, fn in (("DFS", nums_dfs), ("BFS", nums_bfs), ("DP", nums_dp), ("nộp", sol.numsSameConsecDiff)):
        t0 = time.perf_counter()
        for _ in range(20):
            got = fn(9, k)
        dt = (time.perf_counter() - t0) / 20 * 1000
        print(f"n=9 k={k} {name:4}: {dt:7.3f} ms, {len(got):5} số")

số kết quả lớn nhất khi n = 9: (1556, 1) | cận lý thuyết 9 * 2^8 = 2304
n=9 k=0 DFS :   0.037 ms,     9 số
n=9 k=0 BFS :   0.027 ms,     9 số
n=9 k=0 DP  :   0.084 ms,     9 số
n=9 k=0 nộp :   0.033 ms,     9 số
n=9 k=1 DFS :   1.151 ms,  1556 số
n=9 k=1 BFS :   0.795 ms,  1556 số
n=9 k=1 DP  :   0.419 ms,  1556 số
n=9 k=1 nộp :   1.106 ms,  1556 số
n=9 k=3 DFS :   0.255 ms,   240 số
n=9 k=3 BFS :   0.204 ms,   240 số
n=9 k=3 DP  :   0.172 ms,   240 số
n=9 k=3 nộp :   0.254 ms,   240 số
n=9 k=5 DFS :   0.142 ms,     9 số
n=9 k=5 BFS :   0.032 ms,     9 số
n=9 k=5 DP  :   0.090 ms,     9 số
n=9 k=5 nộp :   0.034 ms,     9 số


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm), đối chiếu toàn bộ lưới `n`, `k` hợp lệ, cùng các tính chất. Mọi so sánh dùng **sort** vì thứ tự kết quả không quan trọng.

In [19]:
def all_solutions():
    return {"dfs": nums_dfs, "bfs": nums_bfs, "dp": nums_dp,
            "sol_dfs": sol_dfs, "sol_bfs": sol_bfs, "sol_dp": sol_dp, "final": Solution().numsSameConsecDiff}

def check_all(n, k, expected=None, use_brute=True):
    ref = nums_brute(n, k) if use_brute else sorted(nums_dfs(n, k))
    for name, fn in all_solutions().items():
        got = fn(n, k)
        assert sorted(got) == sorted(ref), (name, n, k)
        assert len(set(got)) == len(got), ("trùng", name, n, k)
    if expected is not None:
        assert sorted(ref) == sorted(expected), ("kỳ vọng sai", n, k, expected, sorted(ref))
    return sorted(ref)

# 12.1 ví dụ đề
for n, k, expected in examples:
    check_all(n, k, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (2, 0, [11, 22, 33, 44, 55, 66, 77, 88, 99]),                      # k = 0: chỉ số lặp chữ số
    (3, 0, [111, 222, 333, 444, 555, 666, 777, 888, 999]),
    (2, 9, [90]),                                                      # chỉ 9 -> 0 (1 -> 10 không có vì 1 + 9 = 10)
    (3, 9, [909]),                                                     # 9 -> 0 -> 9
    (4, 9, [9090]),
    (2, 8, [19, 80, 91]),                                              # 1 -> 9, 8 -> 0, 9 -> 1
    (2, 5, [16, 27, 38, 49, 50, 61, 72, 83, 94]),  # 1->6, 2->7, 3->8, 4->9, 5->0, 6->1, 7->2, 8->3, 9->4
    (2, 3, [14, 25, 30, 36, 41, 47, 52, 58, 63, 69, 74, 85, 96]),
    (3, 4, None),
    (9, 9, [909090909]),                                               # chỉ 9 0 9 0 ...
    (5, 9, [90909]),
]
for n, k, expected in edge:
    check_all(n, k, expected, use_brute=(n <= 5))
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 11 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên toàn lưới n, k rồi xác nhận lời giải đúng không bị lỗi)
def wrong_dup(n, k):                       # hai nhánh +k, -k riêng: k = 0 sinh số lặp
    return nums_dup(n, k)

def wrong_leading_zero(n, k):              # chữ số đầu cho phép 0 (range(0, 10)): sinh số có ít hơn n chữ số
    res = []
    def dfs(num, last, left):
        if left == 0:
            res.append(num)
            return
        for d in {last - k, last + k}:
            if 0 <= d <= 9:
                dfs(num * 10 + d, d, left - 1)
    for f in range(0, 10):
        dfs(f, f, n - 1)
    return res

def wrong_no_bound(n, k):                  # quên kiểm tra 0 <= d <= 9
    res = []
    def dfs(num, last, left):
        if left == 0:
            res.append(num)
            return
        for d in {last - k, last + k}:
            dfs(num * 10 + d, d, left - 1)
    for f in range(1, 10):
        dfs(f, f, n - 1)
    return res

def wrong_plus_only(n, k):                 # chỉ thử hướng +k
    res = []
    def dfs(num, last, left):
        if left == 0:
            res.append(num)
            return
        d = last + k
        if d <= 9:
            dfs(num * 10 + d, d, left - 1)
    for f in range(1, 10):
        dfs(f, f, n - 1)
    return res

def wrong_range_9(n, k):                   # range(1, 9) bỏ sót chữ số đầu 9
    res = []
    def dfs(num, last, left):
        if left == 0:
            res.append(num)
            return
        for d in {last - k, last + k}:
            if 0 <= d <= 9:
                dfs(num * 10 + d, d, left - 1)
    for f in range(1, 9):
        dfs(f, f, n - 1)
    return res

def find_counter(wrong):
    for n in range(2, 6):
        for k in range(10):
            if sorted(wrong(n, k)) != sorted(nums_brute(n, k)):
                return n, k
    return None

counters = {}
for name, fn in (("hai nhánh trùng khi k = 0", wrong_dup), ("cho phép số 0 đứng đầu", wrong_leading_zero),
                 ("quên kiểm tra biên 0..9", wrong_no_bound), ("chỉ thử hướng +k", wrong_plus_only),
                 ("range(1, 9) bỏ sót chữ số 9", wrong_range_9)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    n, k = case
    print(f"{name:30} (n, k) = {case}: sai -> {sorted(fn(n, k))[:8]}... ({len(fn(n, k))} số); đúng -> {len(nums_brute(n, k))} số")
    check_all(n, k)
print("12.3 phản ví dụ: OK")

hai nhánh trùng khi k = 0      (n, k) = (2, 0): sai -> [11, 11, 22, 22, 33, 33, 44, 44]... (18 số); đúng -> 9 số
cho phép số 0 đứng đầu         (n, k) = (2, 0): sai -> [0, 11, 22, 33, 44, 55, 66, 77]... (10 số); đúng -> 9 số
quên kiểm tra biên 0..9        (n, k) = (2, 1): sai -> [10, 12, 21, 23, 32, 34, 43, 45]... (18 số); đúng -> 17 số
chỉ thử hướng +k               (n, k) = (2, 1): sai -> [12, 23, 34, 45, 56, 67, 78, 89]... (8 số); đúng -> 17 số
range(1, 9) bỏ sót chữ số 9    (n, k) = (2, 0): sai -> [11, 22, 33, 44, 55, 66, 77, 88]... (8 số); đúng -> 9 số
12.3 phản ví dụ: OK


In [21]:
# 12.4 toàn bộ lưới n = 2..9, k = 0..9: các cách khớp nhau và khớp bộ đếm; brute cho n <= 5
for n in range(2, 10):
    for k in range(10):
        ref = sorted(nums_dfs(n, k))
        for name, fn in all_solutions().items():
            assert sorted(fn(n, k)) == ref, (name, n, k)
        assert len(ref) == len(set(ref)) == count_dp(n, k)
print("12.4 lưới n = 2..9, k = 0..9 (80 ca): OK")

# 12.5 tính chất của đáp án (kiểm tra bằng định nghĩa)
for n in range(2, 10):
    for k in range(10):
        res = sol.numsSameConsecDiff(n, k)
        for x in res:
            s = str(x)
            assert len(s) == n and s[0] != "0"
            assert all(abs(int(s[i]) - int(s[i + 1])) == k for i in range(n - 1))
        if k == 0:
            assert sorted(res) == [int(str(d) * n) for d in range(1, 10)]
        assert len(res) == count_dp(n, k) >= 1                 # luôn có ít nhất một số (9, 9 - k, 9, ...)
print("12.5 tính chất (độ dài, số 0 đầu, hiệu liền kề, k = 0): OK")

# 12.6 so với oracle trên lấy mẫu ngẫu nhiên theo định nghĩa: số ngẫu nhiên n chữ số thuộc đáp án khi và chỉ khi thỏa định nghĩa
rng = random.Random(6967)
for _ in range(3000):
    n, k = rng.randint(2, 9), rng.randint(0, 9)
    res = set(sol.numsSameConsecDiff(n, k))
    # lấy mẫu: nửa số ngẫu nhiên, nửa số "gần hợp lệ" (đi theo ±k nhưng đôi khi lệch)
    if rng.random() < 0.5:
        x = rng.randint(10 ** (n - 1), 10 ** n - 1)
    else:
        digs = [rng.randint(1, 9)]
        for _ in range(n - 1):
            nd = digs[-1] + rng.choice([-k, k])
            if rng.random() < 0.1 or not 0 <= nd <= 9:
                nd = rng.randint(0, 9)
            digs.append(nd)
        x = int("".join(map(str, digs)))
        if len(str(x)) != n:
            continue
    s = str(x)
    valid = all(abs(int(s[i]) - int(s[i + 1])) == k for i in range(n - 1))
    assert (x in res) == valid, (n, k, x)
print("12.6 lấy mẫu 3000 số theo định nghĩa: thuộc đáp án khi và chỉ khi hợp lệ: OK")

12.4 lưới n = 2..9, k = 0..9 (80 ca): OK


12.5 tính chất (độ dài, số 0 đầu, hiệu liền kề, k = 0): OK


12.6 lấy mẫu 3000 số theo định nghĩa: thuộc đáp án khi và chỉ khi hợp lệ: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — `k = 0` sinh số lặp

Duyệt hai nhánh `last - k` và `last + k` riêng biệt: khi `k = 0` cả hai đều là `last`, mỗi bước nhân đôi và đáp án chứa nhiều bản sao của mỗi số.

In [22]:
n, k = counters["hai nhánh trùng khi k = 0"]
bad, good = wrong_dup(n, k), Solution().numsSameConsecDiff(n, k)
print(f"n={n}, k={k}: sai -> {len(bad)} số (có lặp {len(bad) - len(set(bad))}), đúng -> {len(good)} số")
assert len(bad) > len(good) == len(set(good)) and sorted(set(bad)) == sorted(good)

n=2, k=0: sai -> 18 số (có lặp 9), đúng -> 9 số


Vì sao sai: đề yêu cầu "tất cả các số" (mỗi số một lần). **Đúng:** dùng `set({last - k, last + k})` hoặc `if k == 0: chỉ một nhánh`.

### Lỗi 2 — Cho phép số 0 đứng đầu

Cho chữ số đầu chạy `range(0, 10)` sẽ sinh các "số" như `07` (thực chất là `7`, chỉ có 1 chữ số).

In [23]:
n, k = counters["cho phép số 0 đứng đầu"]
bad, good = wrong_leading_zero(n, k), Solution().numsSameConsecDiff(n, k)
extra = sorted(set(bad) - set(good))
print(f"n={n}, k={k}: sai thừa các số {extra}; độ dài thật của chúng:", [len(str(x)) for x in extra])
assert extra and all(len(str(x)) < n for x in extra)

n=2, k=0: sai thừa các số [0]; độ dài thật của chúng: [1]


Vì sao sai: số có chữ số đầu `0` không phải số `n` chữ số. **Đúng:** chữ số đầu chạy `range(1, 10)`.

### Lỗi 3 — Quên kiểm tra biên `0..9`

Không loại `d < 0` hoặc `d > 9` sẽ sinh ra "chữ số" như `10` hoặc `-1`, làm hỏng số.

In [24]:
n, k = counters["quên kiểm tra biên 0..9"]
bad, good = wrong_no_bound(n, k), Solution().numsSameConsecDiff(n, k)
print(f"n={n}, k={k}: sai -> {len(bad)} số (ví dụ {sorted(bad)[:5]}...), đúng -> {len(good)} số")
assert sorted(bad) != sorted(good)

n=2, k=1: sai -> 18 số (ví dụ [10, 12, 21, 23, 32]...), đúng -> 17 số


Vì sao sai: chữ số phải thuộc `0..9`. **Đúng:** `if 0 <= d <= 9`.

### Lỗi 4 — Chỉ thử hướng `+k`

Nếu chỉ thử `last + k`, bạn bỏ sót mọi số đi **xuống** (như `98`, hoặc bước `8 -> 1` trong `181`).

In [25]:
n, k = counters["chỉ thử hướng +k"]
bad, good = wrong_plus_only(n, k), Solution().numsSameConsecDiff(n, k)
print(f"n={n}, k={k}: sai -> {sorted(bad)}, đúng -> {sorted(good)}")
assert sorted(bad) != sorted(good) and set(bad) < set(good)

n=2, k=1: sai -> [12, 23, 34, 45, 56, 67, 78, 89], đúng -> [10, 12, 21, 23, 32, 34, 43, 45, 54, 56, 65, 67, 76, 78, 87, 89, 98]


Vì sao sai: hiệu tuyệt đối cho phép cả hai chiều. **Đúng:** thử `last - k` và `last + k`.

### Lỗi 5 — `range(1, 9)` bỏ sót chữ số đầu `9`

Lỗi lệch một đơn vị kinh điển của `range`: điểm cuối bị loại.

In [26]:
n, k = counters["range(1, 9) bỏ sót chữ số 9"]
bad, good = wrong_range_9(n, k), Solution().numsSameConsecDiff(n, k)
missing = sorted(set(good) - set(bad))
print(f"n={n}, k={k}: sai thiếu {missing} (đều bắt đầu bằng 9)")
assert missing and all(str(x)[0] == "9" for x in missing)

n=2, k=0: sai thiếu [99] (đều bắt đầu bằng 9)


Vì sao sai: `range(1, 9)` chỉ cho `1..8`. **Đúng:** `range(1, 10)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "sinh mọi chuỗi/số thỏa ràng buộc giữa hai phần tử liền kề, kích thước nhỏ (`n <= 9`)" ⇒ **quay lui** / **BFS theo tầng**; số kết quả chỉ tăng như `2^n`.
- Khi mỗi bước có "`+x` hoặc `-x`", nhớ trường hợp `x = 0` (hai nhánh trùng nhau): dùng `set` hoặc `if x == 0`.
- Số có `n` chữ số: chữ số đầu `1..9`, sau đó `0..9`.
- Xây số bằng `num * 10 + d`, lấy chữ số cuối bằng `num % 10` (không cần chuyển sang chuỗi).
- Muốn **đếm** (không liệt kê) thì dùng DP đếm theo chữ số cuối: `O(n * 10)`.
- Cùng họ: Stepping Numbers (1215, `k = 1`, khoảng giá trị), Letter Combinations of a Phone Number (17), Generate Parentheses (22).
- Test nên có: `k = 0`, `k = 9`, `n` nhỏ nhất (`2`) và lớn nhất (`9`). Đáp án **không bao giờ rỗng** với `n >= 2` (chữ số đầu `9` luôn đi được `9, 9 - k, 9, ...`).

## 15. Biến thể

1. **Chỉ đếm** số lượng (không liệt kê): DP đếm theo chữ số cuối, `O(10 * n)` (`count_dp`, bài tập 5).
2. **Cho phép số 0 đứng đầu** (chuỗi chữ số độ dài `n`): chữ số đầu chạy `0..9`.
3. **Hiệu liền kề thuộc một tập** (ví dụ `{1, 2}`): thay `{last - k, last + k}` bằng `{last + s for s in ±S}`.
4. Bài liên quan: 1215 (Stepping Numbers), 17, 22.

Biến thể 2 (cho phép số 0 đứng đầu, kết quả là chuỗi), kiểm bằng oracle duyệt mọi chuỗi bằng `itertools.product`:

In [27]:
def strings_with_diff(n: int, k: int) -> List[str]:
    res = []
    def dfs(s, left):
        if left == 0:
            res.append(s)
            return
        last = int(s[-1])
        for d in {last - k, last + k}:
            if 0 <= d <= 9:
                dfs(s + str(d), left - 1)
    for f in range(10):
        dfs(str(f), n - 1)
    return res

def strings_brute(n, k):
    return ["".join(map(str, t)) for t in product(range(10), repeat=n)
            if all(abs(t[i] - t[i + 1]) == k for i in range(n - 1))]

for n in range(2, 6):
    for k in range(10):
        assert sorted(strings_with_diff(n, k)) == sorted(strings_brute(n, k)), (n, k)
print(strings_with_diff(2, 9), "(chuỗi 2 chữ số, hiệu 9: có cả '09')")

['09', '90'] (chuỗi 2 chữ số, hiệu 9: có cả '09')


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt mọi số | Kiểm định nghĩa | `O(10^n * n)` | `O(1)` | Oracle, `n` nhỏ |
| 2. DFS | Sinh theo `last ± k` | `O(2^n * n)` | `O(n)` | **Cách nộp chính** |
| 3. BFS | Mở rộng từng tầng | `O(2^n * n)` | `O(2^n)` | Tránh đệ quy |
| 4. DP chiều dài | Dùng lại danh sách con | `O(2^n)` | `O(2^n)` | Khi cần cấu trúc con |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(sorted(Solution().numsSameConsecDiff(2, 9)))
print(sorted(Solution().numsSameConsecDiff(3, 0))[:3])
print(len(Solution().numsSameConsecDiff(2, 2)))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None
my_guess_2 = None
my_guess_3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 3`, `k = 5`: liệt kê tất cả các số hợp lệ (viết dưới dạng danh sách đã sắp xếp tăng dần).

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DFS.

In [30]:
def exercise_3(n: int, k: int) -> List[int]:
    res = []

    def dfs(num: int, last: int, left: int) -> None:
        # TODO 1: điều kiện dừng khi đã đủ n chữ số
        # TODO 2: duyệt d trong {last - k, last + k}, kiểm tra 0 <= d <= 9 rồi gọi đệ quy
        pass

    # TODO 3: thử chữ số đầu từ 1 đến 9
    return res


# for n, k, expected in examples:
#     assert sorted(exercise_3(n, k)) == sorted(expected)

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, tự viết `exercise_4(n, k)` bằng **BFS theo tầng** (danh sách các số hiện có, mỗi vòng thêm một chữ số).

In [31]:
def exercise_4(n: int, k: int) -> List[int]:
    # TODO
    pass


# for n, k, expected in examples:
#     assert sorted(exercise_4(n, k)) == sorted(expected)

### Bài 5 — Mini challenge: đếm không liệt kê (khó)

Viết `exercise_5(n, k)` trả về **số lượng** các số `n` chữ số thỏa điều kiện, trong `O(n * 10)` thời gian, **không** sinh các số.

```python
exercise_5(3, 7) -> 5
exercise_5(2, 1) -> 17
exercise_5(2, 0) -> 9
exercise_5(9, 0) -> 9
```
Gợi ý: `cnt[d]` = số cách đi tiếp từ chữ số `d` cho phần còn lại.

In [32]:
def exercise_5(n: int, k: int) -> int:
    # TODO
    pass


# assert exercise_5(3, 7) == 5
# assert exercise_5(2, 1) == 17
# assert exercise_5(2, 0) == 9
# assert exercise_5(9, 0) == 9

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `n = 2`, `k = 9`: chữ số đầu `d` có chữ số sau `d + 9` hoặc `d - 9` trong `0..9`.
- Mức 2: chỉ `9 -> 0` thỏa (`1 + 9 = 10` quá lớn).
- Mức 3: `[90]`, `[111, 222, 333]`, và `15` (xem giải thích dưới đáp án Bài 1).

**Bài 2**
- Mức 1: chữ số kế tiếp là `last ± 5`.
- Mức 2: từ `1 -> 6 -> 1`, `2 -> 7 -> 2`, `3 -> 8 -> 3`, `4 -> 9 -> 4`, `5 -> 0 -> 5`, `6 -> 1 -> 6`, `7 -> 2 -> 7`, `8 -> 3 -> 8`, `9 -> 4 -> 9`.
- Mức 3: `[161, 272, 383, 494, 505, 616, 727, 838, 949]`.

**Bài 3**
- Mức 1: `if left == 0: res.append(num); return`.
- Mức 2: `for d in {last - k, last + k}: if 0 <= d <= 9: dfs(num * 10 + d, d, left - 1)`.
- Mức 3: `for first in range(1, 10): dfs(first, first, n - 1)`.

**Bài 4**
- Mức 1: `level = list(range(1, 10))`.
- Mức 2: lặp `n - 1` lần, với mỗi `num`, `last = num % 10`.
- Mức 3: `nxt.append(num * 10 + d)` rồi `level = nxt`.

**Bài 5**
- Mức 1: `cnt = [1] * 10` (còn 0 chữ số phải thêm).
- Mức 2: lặp `n - 1` lần: `new[d] = sum(cnt[nd] for nd in {d - k, d + k} if 0 <= nd <= 9)`.
- Mức 3: trả về `sum(cnt[1:])`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(sorted(Solution().numsSameConsecDiff(2, 9)))
print(sorted(Solution().numsSameConsecDiff(3, 0))[:3])
print(len(Solution().numsSameConsecDiff(2, 2)))

assert sorted(Solution().numsSameConsecDiff(2, 9)) == [90]
assert sorted(Solution().numsSameConsecDiff(3, 0))[:3] == [111, 222, 333]
assert len(Solution().numsSameConsecDiff(2, 2)) == 15      # 13,20,24,31,35,42,46,53,57,64,68,75,79,86,97

[90]
[111, 222, 333]
15


Giải thích `n = 2`, `k = 2`: các cặp `(d, d ± 2)` với `d = 1..9`: `13, 20, 24, 31, 35, 42, 46, 53, 57, 64, 68, 75, 79, 86, 97` (chữ số `1` chỉ đi lên `3` vì `1 - 2 < 0`; chữ số `8` chỉ đi xuống `6` vì `8 + 2 = 10`), tổng cộng `15` số.

### Đáp án Bài 2

Từ mỗi chữ số đầu `d`, chữ số thứ hai `d ± 5` rồi lại `± 5`; vì mỗi chữ số chỉ có đúng một lựa chọn hợp lệ nên số có dạng `d, d ± 5, d`:

In [34]:
hand_answer_solution = [161, 272, 383, 494, 505, 616, 727, 838, 949]
assert sorted(Solution().numsSameConsecDiff(3, 5)) == hand_answer_solution == nums_brute(3, 5)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(n: int, k: int) -> List[int]:
    res = []

    def dfs(num: int, last: int, left: int) -> None:
        if left == 0:
            res.append(num)
            return
        for d in {last - k, last + k}:
            if 0 <= d <= 9:
                dfs(num * 10 + d, d, left - 1)

    for first in range(1, 10):
        dfs(first, first, n - 1)
    return res

for n, k, expected in examples:
    assert sorted(exercise_3_solution(n, k)) == sorted(expected)
for n in range(2, 6):
    for k in range(10):
        assert sorted(exercise_3_solution(n, k)) == nums_brute(n, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(n: int, k: int) -> List[int]:
    level = list(range(1, 10))
    for _ in range(n - 1):
        nxt = []
        for num in level:
            last = num % 10
            for d in {last - k, last + k}:
                if 0 <= d <= 9:
                    nxt.append(num * 10 + d)
        level = nxt
    return level

for n, k, expected in examples:
    assert sorted(exercise_4_solution(n, k)) == sorted(expected)
for n in range(2, 6):
    for k in range(10):
        assert sorted(exercise_4_solution(n, k)) == nums_brute(n, k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(n: int, k: int) -> int:
    cnt = [1] * 10
    for _ in range(n - 1):
        cnt = [sum(cnt[nd] for nd in {d - k, d + k} if 0 <= nd <= 9) for d in range(10)]
    return sum(cnt[1:])

assert exercise_5_solution(3, 7) == 5
assert exercise_5_solution(2, 1) == 17
assert exercise_5_solution(2, 0) == 9
assert exercise_5_solution(9, 0) == 9
for n in range(2, 10):
    for k in range(10):
        assert exercise_5_solution(n, k) == len(sol.numsSameConsecDiff(n, k))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DFS xây số từng chữ số)

```python
class Solution:
    def numsSameConsecDiff(self, n: int, k: int) -> List[int]:
        res = []

        def dfs(num: int, last: int, left: int) -> None:
            if left == 0:                          # đủ n chữ số
                res.append(num)
                return
            for d in {last - k, last + k}:         # k = 0: chỉ còn một nhánh
                if 0 <= d <= 9:                    # biên chữ số
                    dfs(num * 10 + d, d, left - 1)

        for first in range(1, 10):                 # chữ số đầu: 1..9 (không 0)
            dfs(first, first, n - 1)
        return res
```

Công thức nhớ nhanh: chữ số kế = `last ± k` (khử trùng khi `k = 0`), trong `0..9`; chữ số đầu `1..9`.

### Checklist trước khi submit

- [ ] Chữ số đầu chạy `range(1, 10)` (không `0`, không bỏ `9`)?
- [ ] Hai nhánh `last - k` và `last + k` được khử trùng khi `k = 0` (dùng `set`)?
- [ ] Có kiểm tra `0 <= d <= 9`?
- [ ] Số lần thêm chữ số là `n - 1` (không phải `n`)?
- [ ] Khi kiểm thử so sánh sau khi **sort** (thứ tự không quan trọng) và kiểm tra không có số lặp?
- [ ] Đã thử: `k = 0`, `k = 9`, `n = 2`, `n = 9`, các ví dụ 1-2?